In [1]:
import numpy as np
from pathlib import Path
import os
import matplotlib.pyplot as plt
cwd = Path.cwd()
import LEAD as lead
from tqdm.notebook import tqdm
import json


# Important variables
SNRs = np.array([-np.inf,-13,-11,-9,-7,-5,-3])
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [ ]:
# ======================================================
# PARAMETERS
# ======================================================
n_categories = 6
categories = list(range(n_categories))
n_trials = 150
dt = 1.0

# Input definition (same for all categories & trials)
one_input = np.concatenate((np.zeros(75), np.ones(25), np.zeros(150)))
input_series = {cat: np.stack([one_input for _ in range(n_trials)]) for cat in categories}

# Definitions for plots
MODEL_PARAM_KEYS = {
    "Linear": [
        "tau", "process_noise", "measure_noise",
        "w0", "w1", "w2", "w3", "w4", "w5"
    ],
    "NonLinear1": [
        "tau", "process_noise", "measure_noise",
        "w0", "w1", "w2", "w3", "w4", "w5",
        "gain", "threshold"
    ],
    "GainModulation": [
        "tau", "process_noise", "measure_noise",
        "w0", "w1", "w2", "w3", "w4", "w5",
        "g0", "g1", "g2", "g3", "g4", "g5",
        "threshold"
    ]
}

# ======================================================
# SYNTHETIC DATA GENERATION
# ======================================================
def w_fun(cat, regul=2, scale=7.5):
    return (np.exp(cat/regul)-1)/(scale*np.exp(5/regul))

def g_fun(cat, root=6, scale=65):
    return -cat*(cat-root)/scale

def generate_data(model_class, params):
    model = model_class(**params)
    state_series = model.measure_simulations(input_series=input_series)
    return state_series, input_series

# Ground-truth parameters
def random_params(model_type='Linear'):
    params = dict(tau=np.random.uniform(3,20), process_noise=np.random.uniform(0.1,1), measure_noise=np.random.uniform(0,1),
              w0=0, w1=np.random.uniform(0,0.05), w2=np.random.uniform(0.05,0.1), w3=np.random.uniform(0.1,0.2), w4=np.random.uniform(0.2,0.3), w5=np.random.uniform(0.3,0.4))
    
    if model_type == 'NonLinear1':
        params['gain'] = np.random.uniform(0.1, 0.5)
        params['threshold'] = np.random.uniform(0, 1.5)
        params['sharpness'] = 5
        
    elif model_type == 'GainModulation':
        params['threshold'] = np.random.uniform(0, 1.5)
        params['sharpness'] = 5
        w0 = np.random.uniform(0,0.1)
        regul, scale_w = np.random.uniform(1, 3), np.random.uniform(3, 15)
        root, scale_g = np.random.uniform(5, 10), np.random.uniform(55, 65)
        
        for cat in range(6):
            params[f'g{cat}'] = g_fun(cat, root=root, scale=scale_g)
            params[f'w{cat}'] = w0 + w_fun(cat, regul=regul, scale=scale_w)
            
    return params

# ======================================================
# FITTING HELPERS
# ======================================================
def fit_single_dataset(model_type, state_train, input_train):
    """
    Fits the specified model type to the data. 
    Note: GainModulation and NonLinear1 require a linear pre-fit.
    """
    
    # 1. Always fit Linear first (needed as standalone or as init for others)
    linear_fit = lead.fitting_tools.clever_fit_linear(
        state_train=state_train, 
        input_train=input_train, 
        n_loops=2, 
        input_start_index=75, 
        input_stop_index=100
    )
    
    if model_type == "Linear":
        return linear_fit.get_params()
        
    elif model_type == "NonLinear1":
        nonlinear_fit = lead.fitting_tools.clever_fit_nonlinear1(
            linear_prefitted=linear_fit,
            state_train=state_train, 
            input_train=input_train, 
            n_loops=2, 
            input_start_index=75, 
            input_stop_index=100
        )
        return nonlinear_fit.get_params()
        
    elif model_type == "GainModulation":
        gainmodul_fit = lead.fitting_tools.clever_fit_gainmodul(
            linear_prefitted=linear_fit, 
            state_train=state_train, 
            input_train=input_train, 
            n_loops=2, 
            input_start_index=75, 
            input_stop_index=100
        )
        return gainmodul_fit.get_params()
        
    return None

def prepare_data_for_fitting(state_series, input_series):
    """Re-organize to gather baseline windows together (same as in ModelRecovery)."""
    state_train = state_series.copy()
    input_train = input_series.copy()
    
    no_stim_sliced = np.concatenate([state_train[0][:, i*50:(i+1)*50] for i in range(5)])
    pre_stim_sliced = np.concatenate([state_train[cat][:, :50] for cat in range(1, n_categories)])
    state_train[0] = np.concatenate((no_stim_sliced, pre_stim_sliced))
    input_train[0] = np.zeros_like(state_train[0])
    
    return state_train, input_train

# ======================================================
# PLOTTING UPDATER
# ======================================================
def update_plots(history, output_dir):
    """
    Generates scatter plots of Real vs Fitted parameters.
    Saves them to disk without displaying.
    """
    plt.close('all') # Clear memory
    
    for model_name, data in history.items():
        if len(data['true']) == 0:
            continue
            
        keys_to_plot = MODEL_PARAM_KEYS[model_name]
        n_params = len(keys_to_plot)
        
        # Calculate grid size
        cols = 4
        rows = int(np.ceil(n_params / cols))
        
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        axes = axes.flatten()
        
        fig.suptitle(f"Parameter Recovery: {model_name} (N={len(data['true'])})", fontsize=16)
        
        # Convert list of dicts to dict of lists for easier plotting
        true_vals = {k: [d[k] for d in data['true']] for k in keys_to_plot}
        fit_vals = {k: [d[k] for d in data['fitted']] for k in keys_to_plot}
        
        for idx, key in enumerate(keys_to_plot):
            ax = axes[idx]
            x = true_vals[key]
            y = fit_vals[key]
            
            # Scatter plot
            ax.scatter(x, y, alpha=0.6, edgecolors='w', s=40)
            
            # Identity line
            lims = [
                np.min([ax.get_xlim(), ax.get_ylim()]), 
                np.max([ax.get_xlim(), ax.get_ylim()])
            ]
            ax.plot(lims, lims, 'k--', alpha=0.5, zorder=0)
            
            # Correlation
            if len(x) > 1:
                corr = np.corrcoef(x, y)[0,1]
                ax.set_title(f"{key} (r={corr:.2f})")
            else:
                ax.set_title(key)
                
            ax.set_xlabel("True")
            ax.set_ylabel("Fitted")
            ax.grid(True, alpha=0.3)
            
        # Hide empty subplots
        for i in range(idx+1, len(axes)):
            axes[i].axis('off')
            
        plt.tight_layout(rect=[0, 0.03, 1, 0.97])
        save_path = os.path.join(output_dir, f"Recovery_{model_name}.png")
        plt.savefig(save_path)
        plt.close(fig)

# ======================================================
# MAIN RECOVERY SCRIPT
# ======================================================
def run_param_recovery(n_iterations=200, checkpoint_path="ParamRecovery/checkpoints.json"):
    
    output_dir = os.path.dirname(checkpoint_path)
    os.makedirs(output_dir, exist_ok=True)

    # ---- Resuming Logic ----
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r") as f:
            checkpoint = json.load(f)
        
        # Load history
        history = checkpoint["history"]
        done_iter = checkpoint["done_iterations"]
        print(f"Resuming from checkpoint: {done_iter} iterations already completed.")
    else:
        # Initialize structure
        history = {
            "Linear": {"true": [], "fitted": []},
            "NonLinear1": {"true": [], "fitted": []},
            "GainModulation": {"true": [], "fitted": []}
        }
        done_iter = 0

    # ---- Main Loop ----
    # Loop from 'done_iter' up to 'n_iterations'
    for i in tqdm(range(done_iter, n_iterations), initial=done_iter, total=n_iterations):
        
        # 1. LINEAR
        true_params_lin = random_params('Linear')
        state_lin, input_lin = generate_data(lead.model.StratifiedLinear, true_params_lin)
        state_train, input_train = prepare_data_for_fitting(state_lin, input_lin)
        fitted_params_lin = fit_single_dataset("Linear", state_train, input_train)
        
        history["Linear"]["true"].append(true_params_lin)
        history["Linear"]["fitted"].append(fitted_params_lin)
        
        # 2. NONLINEAR
        true_params_nl = random_params('NonLinear1')
        state_nl, input_nl = generate_data(lead.model.StratifiedNonLinear1, true_params_nl)
        state_train, input_train = prepare_data_for_fitting(state_nl, input_nl)
        fitted_params_nl = fit_single_dataset("NonLinear1", state_train, input_train)
        
        history["NonLinear1"]["true"].append(true_params_nl)
        history["NonLinear1"]["fitted"].append(fitted_params_nl)

        # 3. GAIN MODULATION
        true_params_gm = random_params('GainModulation')
        state_gm, input_gm = generate_data(lead.model.StratifiedGainModulation, true_params_gm)
        state_train, input_train = prepare_data_for_fitting(state_gm, input_gm)
        fitted_params_gm = fit_single_dataset("GainModulation", state_train, input_train)
        
        history["GainModulation"]["true"].append(true_params_gm)
        history["GainModulation"]["fitted"].append(fitted_params_gm)

        # ---- Save Checkpoint ----
        checkpoint_data = {
            "history": history,
            "done_iterations": i + 1
        }
        with open(checkpoint_path, "w") as f:
            json.dump(checkpoint_data, f, indent=2)

        # ---- Update Figures (Live Monitoring) ----
        # Updates the PNGs on disk every iteration
        update_plots(history, output_dir)

    print("✅ All parameter recovery iterations completed.")
    return history

# ======================================================
# EXECUTION
# ======================================================

# Settings
save_folder = "ParamRecovery"
checkpoint_file = f"{save_folder}/checkpoints_v2.json"

# Run
history = run_param_recovery(n_iterations=200, checkpoint_path=checkpoint_file)

print(f"Results saved in {save_folder}/")

Resuming from checkpoint: 11 iterations already completed.


  6%|5         | 11/200 [00:00<?, ?it/s]